In [ ]:
import os
import pickle
import sys
import time

root_dir = os.path.abspath("..")
os.environ["PYTHONPATH"] = root_dir + os.pathsep + os.environ.get("PYTHONPATH", "")  # parallel workers import the library too
if root_dir not in sys.path:
    sys.path.append(root_dir)

import matplotlib.pyplot as plt
import numpy as np
from joblib import Parallel, delayed, parallel_config
from scipy.optimize import minimize

from current_setpoints.models.forward_model import Fault, ForwardModel, _build_dq_to_phase_map
from current_setpoints.models.machines import ieee_machine2_trained_neural_flux_and_torque
from current_setpoints.optimization import ActiveSetOptimizer, StaticOptimizer

# Waveform deformation vs Joule loss / target torque trade-off (healthy machine, "_tauR" model).
#   Study A  main grid: speed x torque x smoothness weight rho (problem (9) + rho * sum |i_{n+1} - i_n|^2)
#   Study 1  which dq harmonics carry the saving: trajectory restricted to DC + 10 + 20 + ... (also max torque)
#   Study 2  the deliberate 3rd-harmonic current: with vs without i_d3, i_q3
#   Study 3  sensitivity to the voltage / current limit
#   Study 4  allowed torque ripple: mean node torque = T*, |T_n - T*| <= band * T*   (exploratory, not PDF (9))
#   Study 5  eddy-current index (mean squared induced phase voltage) vs Joule-loss saving, from Study A
QUICK = os.environ.get("WAVEFORM_STUDY_QUICK") == "1"  # tiny grid, smoke test only
RERUN = False  # False: load a study's cached results if present (re-plot only); True: recompute everything

CURR_MAX, VOLT_MAX = 30.0, 13.0
N_GRID = 60
N_THETA = 720  # multiple of N_GRID: nodes exactly uniform
SLSQP_OPTS = {"disp": False, "ftol": 1e-8, "maxiter": 500}
N_JOBS = max(1, (os.cpu_count() or 2) - 2)

SPEEDS = [1500, 1800] if QUICK else list(range(1200, 1801, 50))  # 13 speeds
TORQUE_FRACS = [0.7, 1.0] if QUICK else [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]  # of the static max torque at each speed
RHOS = [0.0, 1e3] if QUICK else [0.0, 1e1, 1e2, 1e3, 1e4, 1e5, 1e6]
FOCUS_SPEEDS = [1800] if QUICK else [1500, 1800]  # Studies 1 and 4
FOCUS_FRACS = [0.8] if QUICK else [0.5, 0.8, 1.0]
HARMONIC_SETS = ({"static (DC)": [0], "DC+10": [0, 10], "all nodes": None} if QUICK else
                 {"static (DC)": [0], "DC+10": [0, 10], "DC+10+20": [0, 10, 20], "DC+10+20+30": [0, 10, 20, 30],
                  "all nodes": None})
RIPPLE_BANDS = [0.0, 0.05] if QUICK else [0.0, 0.01, 0.02, 0.05, 0.10]
HARMONIC_TORQUE_BAND = 0.01  # Study 1: a harmonic-limited trajectory cannot hold T exactly at all 60 nodes
                             # (60 equations, 4..28 unknowns) -> mean torque = T*, +-1 % ripple, for every set
LIMIT_CONFIGS = ({"30 A / 13 V": (30.0, 13.0), "30 A / 14 V": (30.0, 14.0)} if QUICK else
                 {"30 A / 12 V": (30.0, 12.0), "30 A / 13 V": (30.0, 13.0), "30 A / 14 V": (30.0, 14.0),
                  "25 A / 13 V": (25.0, 13.0), "35 A / 13 V": (35.0, 13.0)})
LIMIT_FRACS = [0.8] if QUICK else [0.5, 0.8, 1.0]
THIRD_FRACS = [0.8] if QUICK else [0.5, 0.8, 1.0]  # of the static max WITHOUT 3rd harmonic (feasible for both)

TAG = "quick" if QUICK else "full"
data_dir = os.path.join(root_dir, "data")
figs_dir = os.path.join(root_dir, "figs", f"waveform_tradeoff_study_{TAG}")
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
CONST_MECH_SPEED = 30.0 / (np.pi * ieee_machine2_trained_neural_flux_and_torque().n_ppairs)


def rpm_to_omega(rpm):
    return rpm / CONST_MECH_SPEED


def run_parallel(fn, args_list, label):
    t0 = time.time()
    with parallel_config(backend="loky", inner_max_num_threads=1):
        out = Parallel(n_jobs=N_JOBS)(delayed(fn)(*a) for a in args_list)
    n_fail = sum(1 for r in out if not r.get("ok", False))
    print(f"{label}: {len(out)} runs in {time.time() - t0:.0f} s, {n_fail} not solved", flush=True)
    return out


def cached(name, compute):
    path = os.path.join(data_dir, f"waveform_tradeoff_{name}_{TAG}.pkl")
    if os.path.exists(path) and not RERUN:
        with open(path, "rb") as f:
            print(f"{name}: loaded cached results {path}")
            return pickle.load(f)
    res = compute()
    with open(path, "wb") as f:
        pickle.dump(res, f)
    return res


print(f"QUICK={QUICK}, {N_JOBS} parallel jobs, figures -> {figs_dir}")

In [ ]:
class NoThirdHarmonic(ForwardModel):
    """Forward model with i_d3 = i_q3 = 0 added as equality constraints (sinusoidal phase currents only).
    Used by StaticOptimizer (extra_constraints) and by ActiveSetOptimizer (extra_constraints_at_theta, every node)."""

    def extra_constraints(self):
        return super().extra_constraints() + _no_third()

    def extra_constraints_at_theta(self, theta_idx):
        return super().extra_constraints_at_theta(theta_idx) + _no_third()


def _no_third():
    return [{"type": "eq", "fun": lambda i: float(i[2])}, {"type": "eq", "fun": lambda i: float(i[3])}]


def make_fwd(curr_max=CURR_MAX, volt_max=VOLT_MAX, no_third=False):
    drive = ieee_machine2_trained_neural_flux_and_torque(curr_max=curr_max, volt_max=volt_max)
    return (NoThirdHarmonic if no_third else ForwardModel)(drive, n_theta=N_THETA, fault=Fault(()))


M_SEG = 24  # evaluation points per segment


def node_angles(n=N_GRID):
    return np.arange(n) * 2 * np.pi / n


def waveform_metrics(fwd, omega, X):
    """Metrics of a trajectory X (N, 4) given at the node angles (or a constant (4,) current), current linear
    between the nodes (as the drive would apply it):
      copper      mean over the period of sum_k i_k^2 (phase currents) -- Joule loss / R
      dq_ripple   RMS deviation of the node currents from their mean, % of the fundamental amplitude
      thd_extra   phase-current distortion beyond the 1st and 3rd harmonic, % of the fundamental
      third_share 3rd-harmonic phase-current amplitude, % of the fundamental
      eddy_index  mean over the period of sum_k e_k^2, e = w H (J psi + L di/dtheta) the induced phase voltage
                  (proportional to eddy-current loss; uncalibrated -- compare relative to static)
      v_peak / i_peak  true peak phase voltage / current along the whole trajectory"""
    drive = fwd.drive
    X = np.atleast_2d(np.asarray(X, dtype=float))
    if X.shape[0] == 1:
        X = np.tile(X, (N_GRID, 1))
    N = X.shape[0]
    s = np.arange(M_SEG) / M_SEG
    th = ((np.arange(N)[:, None] + s[None, :]) * 2 * np.pi / N).ravel()
    dX = np.roll(X, -1, axis=0) - X
    I = (X[:, None, :] + s[None, :, None] * dX[:, None, :]).reshape(-1, 4)
    didth = np.repeat(dX / (2 * np.pi / N), M_SEG, axis=0)
    Hd = _build_dq_to_phase_map(tuple(range(5)), th)
    J = drive.flux.cross_coupling
    e_dq = np.empty_like(I)
    for t in range(len(I)):
        e_dq[t] = omega * (J @ drive.flux.flux(omega, I[t]) + drive.flux.inductance(omega, I[t]) @ didth[t])
    v_dq = I @ drive.R_stat.T + e_dq
    iph = np.einsum("tpk,tk->tp", Hd, I)
    vph = np.einsum("tpk,tk->tp", Hd, v_dq)
    eph = np.einsum("tpk,tk->tp", Hd, e_dq)
    mean_dq = X.mean(axis=0)
    fund = np.hypot(mean_dq[0], mean_dq[1])
    A = np.abs(np.fft.rfft(iph[:, 0])) * 2 / len(th)
    h = np.arange(len(A))
    extra = (h >= 2) & (h != 3)
    node_T = np.array([drive.torque(omega, x) for x in X])
    return {
        "copper": float(np.mean(np.sum(iph**2, axis=1))),
        "dq_ripple": float(np.sqrt(np.mean(np.sum((X - mean_dq) ** 2, axis=1))) / fund * 100),
        "thd_extra": float(np.sqrt(np.sum(A[extra] ** 2)) / A[1] * 100),
        "third_share": float(A[3] / A[1] * 100),
        "eddy_index": float(np.mean(np.sum(eph**2, axis=1))),
        "v_peak": float(np.abs(vph).max()),
        "i_peak": float(np.abs(iph).max()),
        "torque_mean": float(node_T.mean()),
        "torque_ripple_pp": float((node_T.max() - node_T.min()) / max(abs(node_T.mean()), 1e-9) * 100),
    }


import textwrap

NOTE_BASE = ("Static: constant dq current over the electrical period. Dynamic: the dq current changes with the rotor angle "
             "(PDF problem (9), 60 nodes per period). Joule loss: mean over the period of the sum of the squared phase currents "
             "(times the phase resistance = loss in W); saving = how much less Joule loss dynamic needs than static at the same "
             "speed and torque. T_max: static max torque at that speed.")
NOTE_RHO = ("rho: smoothness weight, penalizes current changes between neighbouring nodes (rho = 0: plain problem (9), most "
            "variation; rho = 1e6: practically constant current). dq ripple: RMS deviation of the dq current from its mean, in % of "
            "the fundamental amplitude (0 = constant). Distortion beyond 1st+3rd: phase-current harmonics other than the fundamental "
            "and the deliberate 3rd harmonic, in % of the fundamental.")
NOTE_HARM = ("DC: the constant part of the dq current; 'static (DC)' = constant current. DC+10: the dq current may also vary 10 times "
             "per electrical period, which appears as the 9th and 11th harmonic in the phase current (frequency = harmonic x f_el, "
             "f_el = rpm x 8 / 60, e.g. 2.2 / 2.6 kHz at 1800 rpm); +20: 19th / 21st; +30: 29th / 31st; 'all nodes': no restriction. "
             "Torque may deviate +-1 % per node (mean = target): a harmonic-limited current cannot hold it exactly constant at all "
             "60 nodes. The 'static (DC)' saving of ~1-2 % is a node-sampling effect (voltage checked at 60 angles instead of 720).")
NOTE_THIRD = ("Without 3rd harmonic: i_d3 = i_q3 = 0 forced (sinusoidal phase currents). Torque levels are fractions of the static max "
              "torque WITHOUT the 3rd harmonic, so both variants are feasible. 3rd-harmonic content: amplitude of the 3rd harmonic of "
              "the phase current, in % of the fundamental.")
NOTE_LIMITS = ("Limits: peak phase current / peak phase voltage. Left: max torque, static (solid) and dynamic (dashed). Right: Joule-loss "
               "saving dynamic vs static at 50 / 80 / 100 % of each configuration's own static max torque, rho = 0.")
NOTE_BAND = ("Torque ripple band: the torque at each node may deviate by +-band x T* from the target, with the mean over the nodes "
             "equal to T* (exploratory: PDF problem (9) requires exactly T* at every node, i.e. band = 0). Right: highest mean torque "
             "under the same rule.")
NOTE_EDDY = ("Eddy-current index: mean over the period of the sum of the squared induced phase voltages w H (J psi + L di/dtheta), a "
             "rough stand-in for eddy-current iron loss; uncalibrated, only its change vs static is meaningful. Left: one line per "
             "speed / torque point, its points = rho values from 0 (largest Joule-loss saving) to 1e6 (smallest).")
NOTE_WAVE = ("Phase a shown (all phases are phase a shifted by 72 deg). Current runs linearly between the 60 nodes; voltage includes "
             "the dynamic term w L di/dtheta. Spectrum: Fourier transform of the phase a current over one electrical period; "
             "top axis = frequency at this speed.")


def finish(fig, fname, *notes, top=0.95):
    """Puts the definitions under the figure, saves it to figs_dir/fname and shows it."""
    width_chars = max(80, int(fig.get_figwidth() * 15))
    txt = "\n".join(line for note in notes for line in textwrap.wrap(note, width_chars))
    n_lines = txt.count("\n") + 1
    bottom = min(0.4, (0.15 + 0.135 * n_lines) / fig.get_figheight())
    fig.tight_layout(rect=[0, bottom, 1, top])
    fig.text(0.01, 0.005, txt, fontsize=8, va="bottom", ha="left", color="0.25")
    path = os.path.join(figs_dir, fname)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=120)
    plt.show()

In [ ]:
def fourier_basis(harmonics, n=N_GRID):
    """Columns 1, cos(k theta_n), sin(k theta_n) for the dq harmonics k (sin(n/2 theta_n) = 0 at the nodes: dropped)."""
    th = node_angles(n)
    cols = []
    for k in harmonics:
        cols += [np.ones(n)] if k == 0 else [np.cos(k * th), np.sin(k * th)]
    B = np.column_stack(cols)
    return B[:, np.abs(B).max(axis=0) > 1e-9]


def _dL_dot(drive, w, x, D, h=1e-4):
    out = np.empty((4, 4))
    for k in range(4):
        e = np.zeros(4)
        e[k] = h
        out[:, k] = (drive.flux.inductance(w, x + e) - drive.flux.inductance(w, x - e)) @ D / (2 * h)
    return out


def solve_direct(fwd, omega, T_target=None, harmonics=None, ripple_band=0.0, X0=None):
    """Problem (9) solved directly as one nonlinear program: every node, exact nonlinear voltage at both ends of every
    segment (the voltage the active-set algorithm converges to), no linearization, no active set. Options:
      harmonics    trajectory restricted to these dq harmonics, X = Phi C (None: every node free)
      ripple_band  0: T(i_n) = T* at every node (problem (9));  > 0: mean node torque = T*, |T(i_n) - T*| <= band T*
      T_target     None: maximize the (constant, or mean with band) torque t instead of minimizing current
    Same per-unit scaling as the library's joint solve. X0: start trajectory (N, 4)."""
    drive = fwd.drive
    N, Imax, Vmax = N_GRID, drive.curr_max, drive.volt_max
    Hn = _build_dq_to_phase_map(tuple(range(5)), node_angles(N))
    Hn1 = np.roll(Hn, -1, axis=0)
    Phi = np.eye(N) if harmonics is None else fourier_basis(harmonics, N)
    nb = Phi.shape[1]
    J, R = drive.flux.cross_coupling, drive.R_stat
    c = omega / (2 * np.pi / N)
    maximize = T_target is None
    ts = 1.0 if maximize else max(abs(T_target), 1.0)
    nz = nb * 4 + (1 if maximize else 0)
    if ripple_band == 0 and nb * 4 + (1 if maximize else 0) < N:
        raise ValueError(f"{N} torque equalities but only {nz} unknowns: use ripple_band > 0 with restricted harmonics")
    cache = {}

    def unpack(z):
        X = Phi @ z[: nb * 4].reshape(nb, 4)
        return X, (z[-1] if maximize else T_target)

    def ev(z):
        key = z.tobytes()
        if key not in cache:
            cache.clear()
            X, t = unpack(z)
            psi = np.array([drive.flux.flux(omega, x) for x in X])
            L = np.array([drive.flux.inductance(omega, x) for x in X])
            D = np.roll(X, -1, axis=0) - X
            ss = X @ R.T + omega * psi @ J.T
            Vs = np.einsum("npk,nk->np", Hn, ss + c * np.einsum("nkl,nl->nk", L, D))
            Ve = np.einsum("npk,nk->np", Hn1, np.roll(ss, -1, axis=0) + c * np.einsum("nkl,nl->nk", np.roll(L, -1, axis=0), D))
            Tn = np.array([drive.torque(omega, x) for x in X])
            G = np.array([drive.torque_grad(omega, x) for x in X])
            cache[key] = dict(X=X, t=t, L=L, D=D, Vs=Vs, Ve=Ve, Ic=np.einsum("npk,nk->np", Hn, X), Tn=Tn, G=G)
        return cache[key]

    def to_z(jX, jt=None):  # (rows, N, 4) jacobian w.r.t. X -> w.r.t. z
        out = np.einsum("rnk,nb->rbk", jX, Phi).reshape(jX.shape[0], nb * 4)
        if maximize:
            out = np.hstack([out, np.zeros((len(out), 1)) if jt is None else jt[:, None]])
        return out

    def torque_rows(z):
        e = ev(z)
        dev = (e["Tn"] - e["t"]) / ts
        jX = np.zeros((N, N, 4))
        jX[np.arange(N), np.arange(N)] = e["G"] / ts
        return dev, jX

    def eq_fun(z):
        dev, _ = torque_rows(z)
        return dev if ripple_band == 0 else np.array([dev.mean()])

    def eq_jac(z):
        _, jX = torque_rows(z)
        jt = -np.ones(N) / ts if maximize else None
        if ripple_band == 0:
            return to_z(jX, jt)
        return to_z(jX.mean(axis=0, keepdims=True), None if jt is None else np.array([-1.0 / ts]))

    def ineq_fun(z):
        e = ev(z)
        parts = [(Imax - e["Ic"]).ravel() / Imax, (Imax + e["Ic"]).ravel() / Imax,
                 (Vmax - e["Vs"]).ravel() / Vmax, (Vmax + e["Vs"]).ravel() / Vmax,
                 (Vmax - e["Ve"]).ravel() / Vmax, (Vmax + e["Ve"]).ravel() / Vmax]
        if ripple_band > 0:
            dev, _ = torque_rows(z)
            band = ripple_band * e["t"] / ts
            parts += [band - dev, band + dev]
        return np.concatenate(parts)

    def ineq_jac(z):
        e = ev(z)
        X, L, D = e["X"], e["L"], e["D"]
        JI, JVs, JVe = (np.zeros((N * 5, N, 4)) for _ in range(3))
        for n in range(N):
            m = (n + 1) % N
            r = slice(5 * n, 5 * n + 5)
            JI[r, n] = Hn[n]
            JVs[r, n] += Hn[n] @ (R + omega * J @ L[n] + c * _dL_dot(drive, omega, X[n], D[n]) - c * L[n])
            JVs[r, m] += Hn[n] @ (c * L[n])
            JVe[r, m] += Hn1[n] @ (R + omega * J @ L[m] + c * _dL_dot(drive, omega, X[m], D[n]) + c * L[m])
            JVe[r, n] += Hn1[n] @ (-c * L[m])
        rows = [to_z(-JI / Imax), to_z(JI / Imax), to_z(-JVs / Vmax), to_z(JVs / Vmax), to_z(-JVe / Vmax), to_z(JVe / Vmax)]
        if ripple_band > 0:
            _, jX = torque_rows(z)
            jb = ripple_band / ts * np.ones(N) if maximize else None
            rows += [to_z(-jX, None if jb is None else jb + 1.0 / ts), to_z(jX, None if jb is None else jb - 1.0 / ts)]
        return np.vstack(rows)

    def objective(z):
        X, t = unpack(z)
        cu = float(np.sum(X**2)) / (N * Imax**2)
        return -t / ts + 1e-6 * cu if maximize else cu

    def objective_grad(z):
        X, _ = unpack(z)
        g = to_z(np.broadcast_to(2 * X / (N * Imax**2), (1, N, 4)).copy())[0]
        if maximize:
            g = g * 1e-6
            g[-1] = -1.0 / ts
        return g

    X0 = np.tile(np.asarray(X0, dtype=float), (N, 1)) if np.ndim(X0) == 1 else np.asarray(X0, dtype=float)
    C0 = np.linalg.lstsq(Phi, X0, rcond=None)[0].ravel()
    z0 = np.r_[C0, np.mean([drive.torque(omega, x) for x in X0])] if maximize else C0
    res = minimize(objective, z0, jac=objective_grad, method="SLSQP",
                   constraints=[{"type": "eq", "fun": eq_fun, "jac": eq_jac}, {"type": "ineq", "fun": ineq_fun, "jac": ineq_jac}],
                   options={"ftol": 1e-10, "maxiter": 1000, "disp": False})
    feasible = bool(np.max(np.abs(eq_fun(res.x))) < 1e-5 and np.min(ineq_fun(res.x)) > -1e-5)
    X, t = unpack(res.x)
    return {"ok": feasible, "X": X, "t": float(t), "message": res.message, "nit": int(res.nit)}

In [ ]:
# ---------------- Study A: speed x torque x smoothness weight rho (library active-set optimizer, problem (9))
def task_static_max(rpm, curr_max=CURR_MAX, volt_max=VOLT_MAX, no_third=False):
    fwd = make_fwd(curr_max, volt_max, no_third)
    s = StaticOptimizer(fwd, SLSQP_OPTS).maximize_torque(rpm_to_omega(rpm))
    return {"ok": bool(s.success), "rpm": rpm, "T_max": float(s.torque), "X": s.curr_dq}


def task_dynamic_max(rpm, curr_max=CURR_MAX, volt_max=VOLT_MAX, no_third=False):
    fwd = make_fwd(curr_max, volt_max, no_third)
    d = ActiveSetOptimizer(fwd, opts=SLSQP_OPTS, n_grid=N_GRID).maximize_torque(rpm_to_omega(rpm))
    return {"ok": bool(d.success), "rpm": rpm, "T_max": float(d.torque) if d.success else np.nan, "X": d.curr_dq}


def task_static_min(rpm, T, curr_max=CURR_MAX, volt_max=VOLT_MAX, no_third=False):
    fwd = make_fwd(curr_max, volt_max, no_third)
    w = rpm_to_omega(rpm)
    s = StaticOptimizer(fwd, SLSQP_OPTS).minimize_current(T, w)
    out = {"ok": bool(s.success), "rpm": rpm, "T": T, "X": s.curr_dq}
    if s.success:
        out.update(waveform_metrics(fwd, w, s.curr_dq))
    return out


def task_dynamic_min(rpm, T, rho=0.0, curr_max=CURR_MAX, volt_max=VOLT_MAX, no_third=False):
    fwd = make_fwd(curr_max, volt_max, no_third)
    w = rpm_to_omega(rpm)
    t0 = time.time()
    d = ActiveSetOptimizer(fwd, opts=SLSQP_OPTS, n_grid=N_GRID, rho=rho).minimize_current(T, w)
    out = {"ok": bool(d.success), "rpm": rpm, "T": T, "rho": rho, "X": d.curr_dq, "time": time.time() - t0,
           "fail_reason": d.diagnostics.get("fail_reason")}
    if d.success:
        out.update(waveform_metrics(fwd, w, d.curr_dq))
    return out


def compute_A():
    smax = run_parallel(task_static_max, [(r,) for r in SPEEDS], "A static max")
    tmax = {r["rpm"]: r["T_max"] for r in smax}
    pts = [(rpm, f) for rpm in SPEEDS for f in TORQUE_FRACS]
    stat = run_parallel(task_static_min, [(rpm, f * tmax[rpm]) for rpm, f in pts], "A static")
    dyn = run_parallel(task_dynamic_min, [(rpm, f * tmax[rpm], rho) for rpm, f in pts for rho in RHOS], "A dynamic")
    for r, (rpm, f) in zip(stat, pts):
        r["frac"] = f
    for r, (rpm, f, rho) in zip(dyn, [(rpm, f, rho) for rpm, f in pts for rho in RHOS]):
        r["frac"] = f
    return {"static_max": tmax, "static": stat, "dynamic": dyn}


A = cached("A_main", compute_A)
STAT_A = {(r["rpm"], r["frac"]): r for r in A["static"]}


def rel(r, key):  # change vs static at the same (speed, torque), %
    s = STAT_A.get((r["rpm"], r["frac"]))
    if not (r.get("ok") and s and s.get("ok")):
        return np.nan
    return 100 * (r[key] / s[key] - 1)


print("Joule-loss change vs static [%] at rho = 0 (rows: torque fraction, columns: speed)")
print("        " + " ".join(f"{rpm:>7d}" for rpm in SPEEDS))
for f in TORQUE_FRACS:
    row = [next((rel(r, "copper") for r in A["dynamic"] if r["rpm"] == rpm and r["frac"] == f and r["rho"] == 0), np.nan) for rpm in SPEEDS]
    print(f"  {f:4.0%}  " + " ".join(f"{v:7.2f}" for v in row))

In [ ]:
# Study A figures: trade-off curves (Joule-loss saving vs deformation along rho) and maps over speed x torque
cmap = plt.get_cmap("viridis")
for metric, xlabel in (("dq_ripple", "dq current ripple [% of fundamental]"),
                       ("thd_extra", "phase-current distortion beyond 1st+3rd harmonic [%]")):
    ncol = min(4, len(SPEEDS))
    nrow = int(np.ceil(len(SPEEDS) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.6 * nrow), squeeze=False, sharey=True)
    for ax, rpm in zip(axs.ravel(), SPEEDS):
        for j, f in enumerate(TORQUE_FRACS):
            rr = sorted([r for r in A["dynamic"] if r["rpm"] == rpm and r["frac"] == f and r.get("ok")], key=lambda r: r["rho"])
            if not rr:
                continue
            x = [r[metric] for r in rr]
            y = [-rel(r, "copper") for r in rr]
            ax.plot(x, y, "o-", color=cmap(j / max(1, len(TORQUE_FRACS) - 1)), ms=4, label=f"{f:.0%} T_max")
            if j == len(TORQUE_FRACS) - 1:
                for r, xi, yi in zip(rr, x, y):
                    ax.annotate(f"{r['rho']:g}", (xi, yi), fontsize=6, xytext=(3, 3), textcoords="offset points")
        ax.set_title(f"{rpm} rpm")
        ax.set_xlabel(xlabel, fontsize=8)
        ax.grid(alpha=0.3)
    for ax in axs[:, 0]:
        ax.set_ylabel("Joule-loss saving vs static [%]")
    for ax in axs.ravel()[len(SPEEDS):]:
        ax.axis("off")
    axs.ravel()[0].legend(fontsize=7, title="torque (labels: rho)", title_fontsize=7)
    fig.suptitle(f"Study A: Joule-loss saving vs waveform deformation ({xlabel.split(' [')[0]}); one line per torque level, "
                 f"points along a line = rho values (labels on the 100 % line)")
    finish(fig, f"A_tradeoff_{metric}.png", NOTE_BASE, NOTE_RHO)


def grid_of(fn):
    return np.array([[fn(rpm, f) for rpm in SPEEDS] for f in TORQUE_FRACS])


def best_saving_within(limit):
    def fn(rpm, f):
        rr = [r for r in A["dynamic"] if r["rpm"] == rpm and r["frac"] == f and r.get("ok") and r["dq_ripple"] <= limit]
        return max((-rel(r, "copper") for r in rr), default=np.nan)
    return fn


def at_rho0(key, relative=True):
    def fn(rpm, f):
        r = next((r for r in A["dynamic"] if r["rpm"] == rpm and r["frac"] == f and r["rho"] == 0), None)
        if r is None or not r.get("ok"):
            return np.nan
        return -rel(r, key) if relative else r[key]
    return fn


maps = [("Joule-loss saving at rho = 0 [%]", grid_of(at_rho0("copper"))),
        ("dq ripple at rho = 0 [%]", grid_of(at_rho0("dq_ripple", False))),
        ("distortion beyond 1st+3rd at rho = 0 [%]", grid_of(at_rho0("thd_extra", False))),
        ("best Joule-loss saving with dq ripple <= 1 % [%]", grid_of(best_saving_within(1.0))),
        ("best Joule-loss saving with dq ripple <= 2 % [%]", grid_of(best_saving_within(2.0))),
        ("best Joule-loss saving with dq ripple <= 5 % [%]", grid_of(best_saving_within(5.0)))]
fig, axs = plt.subplots(2, 3, figsize=(16, 8.5))
for ax, (title, Z) in zip(axs.ravel(), maps):
    im = ax.imshow(Z, origin="lower", aspect="auto", cmap="Blues")
    ax.set_xticks(range(len(SPEEDS)), SPEEDS)
    ax.set_yticks(range(len(TORQUE_FRACS)), [f"{f:.0%}" for f in TORQUE_FRACS])
    for (i, j), v in np.ndenumerate(Z):
        if np.isfinite(v):
            ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=7)
    ax.set_xlabel("speed [rpm]")
    ax.set_ylabel("torque [% of static max]")
    ax.set_title(title, fontsize=10)
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle("Study A over speed x torque: Joule-loss saving, waveform deformation and the saving kept under a ripple cap")
finish(fig, "A_maps.png", NOTE_BASE, NOTE_RHO)

In [ ]:
# ---------------- Study 1: which dq harmonics carry the saving (trajectory restricted to DC + 10 + 20 + ...)
# dq harmonic k appears in the phase currents at (k-1) and (k+1) times the electrical frequency (fundamental part)
def task_harmonic_min(rpm, T, name, harmonics, X_static):
    fwd = make_fwd()
    w = rpm_to_omega(rpm)
    r = solve_direct(fwd, w, T, harmonics=harmonics, ripple_band=HARMONIC_TORQUE_BAND, X0=X_static)
    out = {"ok": r["ok"], "rpm": rpm, "T": T, "set": name, "X": r["X"], "message": str(r["message"])}
    if r["ok"]:
        out.update(waveform_metrics(fwd, w, r["X"]))
    return out


def task_harmonic_max(rpm, name, harmonics, X_static_max):
    fwd = make_fwd()
    r = solve_direct(fwd, rpm_to_omega(rpm), None, harmonics=harmonics, ripple_band=HARMONIC_TORQUE_BAND, X0=X_static_max)
    return {"ok": r["ok"], "rpm": rpm, "set": name, "T_max": r["t"] if r["ok"] else np.nan, "message": str(r["message"])}


def compute_1():
    smax = run_parallel(task_static_max, [(r,) for r in FOCUS_SPEEDS], "1 static max")
    tmax = {r["rpm"]: r for r in smax}
    pts = [(rpm, f) for rpm in FOCUS_SPEEDS for f in FOCUS_FRACS]
    stat = run_parallel(task_static_min, [(rpm, f * tmax[rpm]["T_max"]) for rpm, f in pts], "1 static")
    st = {p: r for p, r in zip(pts, stat)}
    args = [(rpm, f * tmax[rpm]["T_max"], name, hs, st[(rpm, f)]["X"]) for rpm, f in pts for name, hs in HARMONIC_SETS.items()]
    mins = run_parallel(task_harmonic_min, args, "1 restricted min-current")
    for r, (rpm, f, name) in zip(mins, [(rpm, f, name) for rpm, f in pts for name in HARMONIC_SETS]):
        r["frac"] = f
    maxs = run_parallel(task_harmonic_max, [(rpm, name, hs, tmax[rpm]["X"]) for rpm in FOCUS_SPEEDS for name, hs in HARMONIC_SETS.items()],
                        "1 restricted max torque")
    # consistency: the unrestricted direct solve must reproduce the library's active-set result
    rpm0, f0 = pts[0]
    ref = task_dynamic_min(rpm0, f0 * tmax[rpm0]["T_max"])
    fwd0 = make_fwd()
    strict = solve_direct(fwd0, rpm_to_omega(rpm0), f0 * tmax[rpm0]["T_max"], X0=st[(rpm0, f0)]["X"])  # PDF (9), no band
    if strict["ok"]:
        strict.update(waveform_metrics(fwd0, rpm_to_omega(rpm0), strict["X"]))
    return {"static_max": {k: v["T_max"] for k, v in tmax.items()}, "static": st, "min": mins, "max": maxs, "check": ref, "check_direct": strict}


S1 = cached("1_harmonics", compute_1)
chk = S1["check"]
direct_all = S1["check_direct"]
if chk.get("ok") and direct_all.get("ok"):
    print(f"consistency check, strict problem (9) ({chk['rpm']} rpm, {chk['T']:.2f} Nm): Joule loss active set {chk['copper']:.3f}, direct {direct_all['copper']:.3f} "
          f"(difference {100 * (direct_all['copper'] / chk['copper'] - 1):+.3f} %)")
fe = {rpm: rpm * 8 / 60 for rpm in FOCUS_SPEEDS}  # electrical frequency [Hz]
for rpm in FOCUS_SPEEDS:
    print(f"\n{rpm} rpm (f_el = {fe[rpm]:.0f} Hz; dq 10 -> phase 9/11 = {9 * fe[rpm] / 1e3:.1f}/{11 * fe[rpm] / 1e3:.1f} kHz, "
          f"dq 20 -> 19/21 = {19 * fe[rpm] / 1e3:.1f}/{21 * fe[rpm] / 1e3:.1f} kHz, dq 30 -> 29/31 = {29 * fe[rpm] / 1e3:.1f}/{31 * fe[rpm] / 1e3:.1f} kHz)")
    for name in HARMONIC_SETS:
        mx = next((r for r in S1["max"] if r["rpm"] == rpm and r["set"] == name), {})
        line = f"  {name:14s} max torque {mx.get('T_max', np.nan):6.3f} Nm |"
        for f in FOCUS_FRACS:
            r = next((r for r in S1["min"] if r["rpm"] == rpm and r["set"] == name and r["frac"] == f), {})
            s = S1["static"][(rpm, f)]
            sav = -100 * (r["copper"] / s["copper"] - 1) if r.get("ok") and s.get("ok") else np.nan
            line += f" {f:.0%}: saving {sav:5.2f} %, ripple {r.get('dq_ripple', np.nan):5.2f} % |"
        print(line)

fig, axs = plt.subplots(1, 2, figsize=(14, 4.8))
names = list(HARMONIC_SETS)
wbar = 0.8 / (len(FOCUS_SPEEDS) * len(FOCUS_FRACS))
k = 0
for rpm in FOCUS_SPEEDS:
    for f in FOCUS_FRACS:
        vals = []
        for name in names:
            r = next((r for r in S1["min"] if r["rpm"] == rpm and r["set"] == name and r["frac"] == f), {})
            s = S1["static"][(rpm, f)]
            vals.append(-100 * (r["copper"] / s["copper"] - 1) if r.get("ok") and s.get("ok") else np.nan)
        axs[0].bar(np.arange(len(names)) + k * wbar, vals, wbar, label=f"{rpm} rpm, {f:.0%}")
        k += 1
axs[0].set_xticks(np.arange(len(names)) + 0.4 - wbar / 2, names, rotation=15)
axs[0].set_ylabel("Joule-loss saving vs static [%]")
axs[0].legend(fontsize=7)
axs[0].grid(alpha=0.3, axis="y")
for rpm in FOCUS_SPEEDS:
    axs[1].plot(names, [next((r["T_max"] for r in S1["max"] if r["rpm"] == rpm and r["set"] == n), np.nan) for n in names], "o-", label=f"{rpm} rpm")
axs[1].set_ylabel("max torque [Nm]")
axs[1].grid(alpha=0.3)
axs[1].legend()
axs[1].tick_params(axis="x", rotation=15)
fig.suptitle(f"Study 1: trajectory restricted to selected dq harmonics (mean torque = T*, +-{100 * HARMONIC_TORQUE_BAND:g} % torque ripple)")
finish(fig, "1_harmonics.png", NOTE_BASE, NOTE_HARM)

In [ ]:
# ---------------- Study 2: the deliberate 3rd-harmonic current (with vs without i_d3, i_q3)
def compute_2():
    # all runs of one kind submitted together (keeps every core busy during the slow dynamic max-torque probes)
    out = {}
    cases = [(r, no3) for no3 in (False, True) for r in SPEEDS]
    smax = run_parallel(task_static_max, [(r, CURR_MAX, VOLT_MAX, no3) for r, no3 in cases], "2 static max")
    dmax = run_parallel(task_dynamic_max, [(r, CURR_MAX, VOLT_MAX, no3) for r, no3 in cases], "2 dynamic max")
    for no3 in (False, True):
        out[("static_max", no3)] = [x for x, (r, n) in zip(smax, cases) if n == no3]
        out[("dynamic_max", no3)] = [x for x, (r, n) in zip(dmax, cases) if n == no3]
    tmax_no3 = {r["rpm"]: r["T_max"] for r in out[("static_max", True)]}
    pts = [(rpm, f) for rpm in SPEEDS for f in THIRD_FRACS if np.isfinite(tmax_no3[rpm]) and tmax_no3[rpm] > 0.2]
    jobs = [(rpm, f, no3) for no3 in (False, True) for rpm, f in pts]
    st = run_parallel(task_static_min, [(rpm, f * tmax_no3[rpm], CURR_MAX, VOLT_MAX, no3) for rpm, f, no3 in jobs], "2 static")
    dy = run_parallel(task_dynamic_min, [(rpm, f * tmax_no3[rpm], 0.0, CURR_MAX, VOLT_MAX, no3) for rpm, f, no3 in jobs], "2 dynamic")
    for a, b, (rpm, f, no3) in zip(st, dy, jobs):
        a["frac"] = b["frac"] = f
    for no3 in (False, True):
        out[("static", no3)] = [a for a, j in zip(st, jobs) if j[2] == no3]
        out[("dynamic", no3)] = [b for b, j in zip(dy, jobs) if j[2] == no3]
    return out


S2 = cached("2_third_harmonic", compute_2)
fig, axs = plt.subplots(1, 3, figsize=(17, 4.8))
for no3, ls in ((False, "-"), (True, "--")):
    lab = "without 3rd harm." if no3 else "with 3rd harm."
    axs[0].plot(SPEEDS, [r["T_max"] for r in S2[("static_max", no3)]], "o" + ls, color="tab:blue", label=f"static, {lab}")
    axs[0].plot(SPEEDS, [r["T_max"] for r in S2[("dynamic_max", no3)]], "s" + ls, color="tab:red", label=f"dynamic, {lab}")
axs[0].set_xlabel("speed [rpm]"); axs[0].set_ylabel("max torque [Nm]"); axs[0].grid(alpha=0.3); axs[0].legend(fontsize=8)
axs[0].set_title("max torque")
for j, f in enumerate(THIRD_FRACS):
    for mode, mk in (("static", "o"), ("dynamic", "s")):
        w3 = {r["rpm"]: r for r in S2[(mode, False)] if r["frac"] == f}
        n3 = {r["rpm"]: r for r in S2[(mode, True)] if r["frac"] == f}
        rr = [rpm for rpm in SPEEDS if rpm in w3 and w3[rpm].get("ok") and n3[rpm].get("ok")]
        axs[1].plot(rr, [100 * (w3[rpm]["copper"] / n3[rpm]["copper"] - 1) for rpm in rr], mk + "-", color=cmap(j / max(1, len(THIRD_FRACS) - 1)),
                    label=f"{mode}, {f:.0%}", alpha=1.0 if mode == "static" else 0.6)
        axs[2].plot(rr, [w3[rpm]["third_share"] for rpm in rr], mk + "-", color=cmap(j / max(1, len(THIRD_FRACS) - 1)),
                    alpha=1.0 if mode == "static" else 0.6, label=f"{mode}, {f:.0%}")
axs[1].set_xlabel("speed [rpm]"); axs[1].set_ylabel("Joule loss with vs without 3rd harmonic [%]"); axs[1].grid(alpha=0.3); axs[1].legend(fontsize=7)
axs[1].set_title("Joule loss at the same torque (fraction of the no-3rd static max)")
axs[2].set_xlabel("speed [rpm]"); axs[2].set_ylabel("3rd-harmonic phase current [% of fundamental]"); axs[2].grid(alpha=0.3); axs[2].legend(fontsize=7)
axs[2].set_title("deliberate 3rd-harmonic content")
fig.suptitle("Study 2: with vs without the 3rd-harmonic current")
finish(fig, "2_third_harmonic.png", NOTE_BASE, NOTE_THIRD)

In [ ]:
# ---------------- Study 3: sensitivity to the voltage / current limit
def compute_3():
    # all configurations submitted together (keeps every core busy during the slow dynamic max-torque probes)
    cases = [(name, r) for name in LIMIT_CONFIGS for r in SPEEDS]
    smax = run_parallel(task_static_max, [(r, *LIMIT_CONFIGS[n]) for n, r in cases], "3 static max")
    dmax = run_parallel(task_dynamic_max, [(r, *LIMIT_CONFIGS[n]) for n, r in cases], "3 dynamic max")
    tmax = {(n, r): x["T_max"] for x, (n, r) in zip(smax, cases)}
    jobs = [(n, rpm, f) for n in LIMIT_CONFIGS for rpm in SPEEDS for f in LIMIT_FRACS
            if np.isfinite(tmax[(n, rpm)]) and tmax[(n, rpm)] > 0.2]
    st = run_parallel(task_static_min, [(rpm, f * tmax[(n, rpm)], *LIMIT_CONFIGS[n]) for n, rpm, f in jobs], "3 static")
    dy = run_parallel(task_dynamic_min, [(rpm, f * tmax[(n, rpm)], 0.0, *LIMIT_CONFIGS[n]) for n, rpm, f in jobs], "3 dynamic")
    for a, b, (n, rpm, f) in zip(st, dy, jobs):
        a["frac"] = b["frac"] = f
    out = {}
    for name in LIMIT_CONFIGS:
        out[name] = {"static_max": [x for x, (n, r) in zip(smax, cases) if n == name],
                     "dynamic_max": [x for x, (n, r) in zip(dmax, cases) if n == name],
                     "static": [a for a, j in zip(st, jobs) if j[0] == name],
                     "dynamic": [b for b, j in zip(dy, jobs) if j[0] == name]}
    return out


S3 = cached("3_limits", compute_3)
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
colors = plt.get_cmap("tab10")
for j, (name, res) in enumerate(S3.items()):
    axs[0].plot(SPEEDS, [r["T_max"] for r in res["static_max"]], "o-", color=colors(j), label=f"{name} static")
    axs[0].plot(SPEEDS, [r["T_max"] for r in res["dynamic_max"]], "s--", color=colors(j), label=f"{name} dynamic")
    for f, mk in zip(LIMIT_FRACS, ("o", "s", "^", "v")):
        sd = {(r["rpm"]): r for r in res["static"] if r["frac"] == f}
        dd = {(r["rpm"]): r for r in res["dynamic"] if r["frac"] == f}
        rr = [rpm for rpm in SPEEDS if rpm in sd and sd[rpm].get("ok") and dd[rpm].get("ok")]
        axs[1].plot(rr, [-100 * (dd[rpm]["copper"] / sd[rpm]["copper"] - 1) for rpm in rr], mk + "-", color=colors(j),
                    label=f"{name}, {f:.0%}" if True else None, alpha=0.5 + 0.5 * (f == LIMIT_FRACS[-1]))
axs[0].set_xlabel("speed [rpm]"); axs[0].set_ylabel("max torque [Nm]"); axs[0].grid(alpha=0.3); axs[0].legend(fontsize=7, ncol=2)
axs[1].set_xlabel("speed [rpm]"); axs[1].set_ylabel("Joule-loss saving dynamic vs static [%]"); axs[1].grid(alpha=0.3); axs[1].legend(fontsize=6, ncol=2)
fig.suptitle("Study 3: voltage / current limit sensitivity (rho = 0)")
finish(fig, "3_limits.png", NOTE_BASE, NOTE_LIMITS)

In [ ]:
# ---------------- Study 4: allowed torque ripple (mean node torque = T*, |T_n - T*| <= band * T*) -- exploratory
def task_band_min(rpm, T, band, X_static):
    fwd = make_fwd()
    w = rpm_to_omega(rpm)
    r = solve_direct(fwd, w, T, ripple_band=band, X0=X_static)
    out = {"ok": r["ok"], "rpm": rpm, "T": T, "band": band, "X": r["X"], "message": str(r["message"])}
    if r["ok"]:
        out.update(waveform_metrics(fwd, w, r["X"]))
    return out


def task_band_max(rpm, band, X_static_max):
    fwd = make_fwd()
    r = solve_direct(fwd, rpm_to_omega(rpm), None, ripple_band=band, X0=X_static_max)
    return {"ok": r["ok"], "rpm": rpm, "band": band, "T_max": r["t"] if r["ok"] else np.nan, "message": str(r["message"])}


def compute_4():
    smax = run_parallel(task_static_max, [(r,) for r in FOCUS_SPEEDS], "4 static max")
    tmax = {r["rpm"]: r for r in smax}
    pts = [(rpm, f) for rpm in FOCUS_SPEEDS for f in FOCUS_FRACS]
    stat = run_parallel(task_static_min, [(rpm, f * tmax[rpm]["T_max"]) for rpm, f in pts], "4 static")
    st = {p: r for p, r in zip(pts, stat)}
    mins = run_parallel(task_band_min, [(rpm, f * tmax[rpm]["T_max"], b, st[(rpm, f)]["X"]) for rpm, f in pts for b in RIPPLE_BANDS], "4 band min-current")
    for r, (rpm, f, b) in zip(mins, [(rpm, f, b) for rpm, f in pts for b in RIPPLE_BANDS]):
        r["frac"] = f
    maxs = run_parallel(task_band_max, [(rpm, b, tmax[rpm]["X"]) for rpm in FOCUS_SPEEDS for b in RIPPLE_BANDS], "4 band max torque")
    return {"static_max": {k: v["T_max"] for k, v in tmax.items()}, "static": st, "min": mins, "max": maxs}


S4 = cached("4_torque_ripple", compute_4)
fig, axs = plt.subplots(1, 3, figsize=(17, 4.8))
for rpm, mk in zip(FOCUS_SPEEDS, ("o", "s")):
    for j, f in enumerate(FOCUS_FRACS):
        rr = sorted([r for r in S4["min"] if r["rpm"] == rpm and r["frac"] == f and r.get("ok")], key=lambda r: r["band"])
        s = S4["static"][(rpm, f)]
        col = cmap(j / max(1, len(FOCUS_FRACS) - 1))
        axs[0].plot([100 * r["band"] for r in rr], [-100 * (r["copper"] / s["copper"] - 1) for r in rr], mk + "-", color=col, label=f"{rpm} rpm, {f:.0%}")
        axs[1].plot([100 * r["band"] for r in rr], [r["dq_ripple"] for r in rr], mk + "-", color=col, label=f"{rpm} rpm, {f:.0%}")
    mx = sorted([r for r in S4["max"] if r["rpm"] == rpm], key=lambda r: r["band"])
    axs[2].plot([100 * r["band"] for r in mx], [r["T_max"] for r in mx], mk + "-", label=f"{rpm} rpm (mean torque)")
axs[0].set_xlabel("allowed torque ripple band [% of T*]"); axs[0].set_ylabel("Joule-loss saving vs static [%]"); axs[0].legend(fontsize=7); axs[0].grid(alpha=0.3)
axs[1].set_xlabel("allowed torque ripple band [% of T*]"); axs[1].set_ylabel("dq current ripple [%]"); axs[1].grid(alpha=0.3)
axs[2].set_xlabel("allowed torque ripple band [% of mean torque]"); axs[2].set_ylabel("max mean torque [Nm]"); axs[2].legend(fontsize=8); axs[2].grid(alpha=0.3)
fig.suptitle("Study 4: allowing torque ripple (exploratory, not PDF problem (9))")
finish(fig, "4_torque_ripple.png", NOTE_BASE, NOTE_BAND)

In [ ]:
# ---------------- Study 5: eddy-current index vs Joule-loss saving (from Study A; index uncalibrated -> relative to static)
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
for i, rpm in enumerate(SPEEDS):
    for j, f in enumerate(TORQUE_FRACS):
        rr = sorted([r for r in A["dynamic"] if r["rpm"] == rpm and r["frac"] == f and r.get("ok")], key=lambda r: r["rho"])
        if f == TORQUE_FRACS[-1] or f == TORQUE_FRACS[0]:
            axs[0].plot([rel(r, "eddy_index") for r in rr], [-rel(r, "copper") for r in rr], "o-", ms=3,
                        color=plt.get_cmap("plasma")(i / max(1, len(SPEEDS) - 1)), alpha=1.0 if f == TORQUE_FRACS[-1] else 0.4,
                        label=f"{rpm} rpm, {f:.0%}")
axs[0].axvline(0, color="gray", lw=0.8)
axs[0].set_xlabel("eddy-current index change vs static [%]")
axs[0].set_ylabel("Joule-loss saving vs static [%]")
axs[0].set_title("along rho (one line per speed / torque point, rho from 0 to 1e6)")
axs[0].grid(alpha=0.3)
axs[0].legend(fontsize=6, ncol=2)
Z = grid_of(lambda rpm, f: next((rel(r, "eddy_index") for r in A["dynamic"] if r["rpm"] == rpm and r["frac"] == f and r["rho"] == 0 and r.get("ok")), np.nan))
im = axs[1].imshow(Z, origin="lower", aspect="auto", cmap="Reds")
axs[1].set_xticks(range(len(SPEEDS)), SPEEDS)
axs[1].set_yticks(range(len(TORQUE_FRACS)), [f"{f:.0%}" for f in TORQUE_FRACS])
for (i, j), v in np.ndenumerate(Z):
    if np.isfinite(v):
        axs[1].text(j, i, f"{v:+.1f}", ha="center", va="center", fontsize=7)
axs[1].set_title("eddy-current index change vs static at rho = 0 [%]")
axs[1].set_xlabel("speed [rpm]")
axs[1].set_ylabel("torque [% of static max]")
fig.colorbar(im, ax=axs[1], shrink=0.8)
fig.suptitle("Study 5: Joule-loss saving vs induced-voltage (eddy-current) index")
finish(fig, "5_eddy_index.png", NOTE_BASE, NOTE_EDDY)

In [ ]:
# ---------------- Waveform figures from the saved trajectories (no recomputation)
H_MAX = 32  # harmonics shown (up to the 31st: dq harmonic 30)


def dense_waveforms(fwd, omega, X):
    # (theta [deg], phase currents (5, T), phase voltages (5, T), torque (T)) along the trajectory, current linear
    # between the nodes, voltage incl. the dynamic term -- the same evaluation as waveform_metrics.
    drive = fwd.drive
    X = np.atleast_2d(np.asarray(X, dtype=float))
    if X.shape[0] == 1:
        X = np.tile(X, (N_GRID, 1))
    N = X.shape[0]
    s = np.arange(M_SEG) / M_SEG
    th = ((np.arange(N)[:, None] + s[None, :]) * 2 * np.pi / N).ravel()
    dX = np.roll(X, -1, axis=0) - X
    I = (X[:, None, :] + s[None, :, None] * dX[:, None, :]).reshape(-1, 4)
    didth = np.repeat(dX / (2 * np.pi / N), M_SEG, axis=0)
    Hd = _build_dq_to_phase_map(tuple(range(5)), th)
    J = drive.flux.cross_coupling
    v_dq = np.empty_like(I)
    T = np.empty(len(I))
    for t in range(len(I)):
        v_dq[t] = drive.R_stat @ I[t] + omega * (J @ drive.flux.flux(omega, I[t]) + drive.flux.inductance(omega, I[t]) @ didth[t])
        T[t] = drive.torque(omega, I[t])
    return np.rad2deg(th), np.einsum("tpk,tk->pt", Hd, I), np.einsum("tpk,tk->pt", Hd, v_dq), T


def spectrum(i_a):
    return (np.abs(np.fft.rfft(i_a)) * 2 / len(i_a))[:H_MAX]


def style_wave_axes(ax_i, ax_v, curr_max=CURR_MAX, volt_max=VOLT_MAX, xlabel=True):
    for ax, lim, lab in ((ax_i, curr_max, "phase a current [A]"), (ax_v, volt_max, "phase a voltage [V]")):
        ax.axhline(lim, color="gray", ls=":"); ax.axhline(-lim, color="gray", ls=":")
        ax.set_xlim(0, 360); ax.set_ylabel(lab); ax.grid(alpha=0.3)
        if xlabel:
            ax.set_xlabel("rotor angle theta [deg, electrical]")


def style_spectrum(ax, rpm, log=True):
    f_el = rpm * 8 / 60.0
    if log:
        ax.set_yscale("log")
    ax.set_xticks(range(0, H_MAX, 2)); ax.set_xlabel("harmonic order of the phase current"); ax.set_ylabel("amplitude [A]")
    ax.grid(alpha=0.3)
    top = ax.secondary_xaxis("top", functions=(lambda h, f=f_el: h * f / 1000.0, lambda k, f=f_el: k * 1000.0 / f))
    top.set_xlabel("frequency [kHz]")


fwd_plot = make_fwd()
wave_dir = "waveforms"
print("drawing waveform figures ...")

In [ ]:
# Torque ladder per speed (like the earlier waveform study, Study A): static (black dashed) vs dynamic rho = 0 (red)
for rpm in SPEEDS:
    w = rpm_to_omega(rpm)
    fig, axes = plt.subplots(len(TORQUE_FRACS), 3, figsize=(18, 3.3 * len(TORQUE_FRACS)), squeeze=False)
    fig.suptitle(f"Torque ladder, {rpm} rpm (static max {A['static_max'][rpm]:.2f} Nm): static (black dashed) vs dynamic rho = 0 (red)", fontsize=13)
    for r, f in enumerate(TORQUE_FRACS):
        st = STAT_A.get((rpm, f))
        dy = next((x for x in A["dynamic"] if x["rpm"] == rpm and x["frac"] == f and x["rho"] == 0), None)
        ax_i, ax_v, ax_s = axes[r]
        info = []
        for res, col, ls, lab, off in ((st, "k", "--", "static", -0.2), (dy, "tab:red", "-", "dynamic", 0.2)):
            if res is None or not res.get("ok"):
                continue
            th, iph, vph, _ = dense_waveforms(fwd_plot, w, res["X"])
            ax_i.plot(th, iph[0], ls, color=col, lw=1.5, label=lab)
            ax_v.plot(th, vph[0], ls, color=col, lw=1.5, label=lab)
            ax_s.bar(np.arange(H_MAX) + off, spectrum(iph[0]), width=0.4, color=col, label=lab)
        if st and dy and st.get("ok") and dy.get("ok"):
            info = f"Joule-loss saving {-rel(dy, 'copper'):.2f} %, dq ripple {dy['dq_ripple']:.2f} %, distortion beyond 1st+3rd {dy['thd_extra']:.2f} %"
        else:
            info = "not solved in both modes"
        ax_i.set_title(f"T* = {f * A['static_max'][rpm]:.2f} Nm ({f:.0%} of static max): {info}", fontsize=9, loc="left")
        style_wave_axes(ax_i, ax_v, xlabel=(r == len(TORQUE_FRACS) - 1))
        style_spectrum(ax_s, rpm, log=False)
        if r == 0:
            ax_i.legend(fontsize=8); ax_s.legend(fontsize=8)
    finish(fig, os.path.join(wave_dir, f"ladder_{rpm}rpm.png"), NOTE_BASE, NOTE_WAVE, top=0.97)

In [ ]:
# rho sweep per speed (like the earlier waveform study, Study B), at 100 % and 80 % of the static max torque
RHO_FRACS = [f for f in (1.0, 0.8) if f in TORQUE_FRACS]
for rpm in SPEEDS:
    w = rpm_to_omega(rpm)
    for f in RHO_FRACS:
        st = STAT_A.get((rpm, f))
        runs = sorted([x for x in A["dynamic"] if x["rpm"] == rpm and x["frac"] == f and x.get("ok")], key=lambda x: x["rho"])
        if not (st and st.get("ok") and runs):
            continue
        fig, ((ax_i, ax_v), (ax_s, ax_c)) = plt.subplots(2, 2, figsize=(16, 9.5))
        fig.suptitle(f"rho sweep, {rpm} rpm, T* = {f * A['static_max'][rpm]:.2f} Nm ({f:.0%} of static max): dynamic for increasing rho "
                     f"(colours), static black dashed", fontsize=13)
        colors = plt.cm.viridis(np.linspace(0.0, 0.9, len(runs)))
        offs = np.linspace(-0.3, 0.3, len(runs) + 1)
        th, iph, vph, _ = dense_waveforms(fwd_plot, w, st["X"])
        ax_i.plot(th, iph[0], "k--", lw=1.2, label="static"); ax_v.plot(th, vph[0], "k--", lw=1.2)
        sp = spectrum(iph[0]); ok = sp > 1e-6
        ax_s.plot(np.arange(H_MAX)[ok] + offs[-1], sp[ok], "kx", ms=7, label="static")
        for k, (x, col) in enumerate(zip(runs, colors)):
            th, iph, vph, _ = dense_waveforms(fwd_plot, w, x["X"])
            ax_i.plot(th, iph[0], color=col, lw=1.3, label=f"rho = {x['rho']:g}")
            ax_v.plot(th, vph[0], color=col, lw=1.3)
            sp = spectrum(iph[0]); ok = sp > 1e-6
            ax_s.plot(np.arange(H_MAX)[ok] + offs[k], sp[ok], "o", color=col, ms=4, label=f"rho = {x['rho']:g}")
        style_wave_axes(ax_i, ax_v)
        ax_i.legend(fontsize=8, ncol=2)
        style_spectrum(ax_s, rpm)
        ax_s.legend(fontsize=7, ncol=2)
        ax_s.set_title("Fourier transform of the phase a current (amplitudes below 1e-6 A not drawn)", fontsize=10)
        sav = [-rel(x, "copper") for x in runs]
        bars = ax_c.bar([f"{x['rho']:g}" for x in runs], sav, color=colors)
        ax_c.bar_label(bars, labels=[f"{v:.2f} %\n{x['dq_ripple']:.2f} %" for v, x in zip(sav, runs)], fontsize=8)
        ax_c.set_ylim(top=1.2 * np.nanmax(sav))  # room for the two-line bar labels
        ax_c.set_xlabel("rho"); ax_c.set_ylabel("Joule-loss saving vs static [%]"); ax_c.grid(axis="y", alpha=0.3)
        ax_c.set_title("Joule-loss saving vs static (upper number) and dq ripple (lower number)", fontsize=10)
        finish(fig, os.path.join(wave_dir, f"rho_{rpm}rpm_{f * 100:.0f}pct.png"), NOTE_BASE, NOTE_RHO, NOTE_WAVE)

In [ ]:
# Study 1 waveforms: the trajectory restricted to the dq harmonics
for rpm in FOCUS_SPEEDS:
    w = rpm_to_omega(rpm)
    for f in FOCUS_FRACS:
        st = S1["static"][(rpm, f)]
        runs = [next((x for x in S1["min"] if x["rpm"] == rpm and x["frac"] == f and x["set"] == n), None) for n in HARMONIC_SETS]
        fig, ((ax_i, ax_v), (ax_s, ax_c)) = plt.subplots(2, 2, figsize=(16, 9.5))
        fig.suptitle(f"Restricted harmonics, {rpm} rpm, T* = {f * S1['static_max'][rpm]:.2f} Nm ({f:.0%} of static max), "
                     f"static black dashed", fontsize=13)
        colors = plt.cm.plasma(np.linspace(0.0, 0.85, len(HARMONIC_SETS)))
        offs = np.linspace(-0.3, 0.3, len(HARMONIC_SETS) + 1)
        th, iph, vph, _ = dense_waveforms(fwd_plot, w, st["X"])
        ax_i.plot(th, iph[0], "k--", lw=1.2, label="static optimizer"); ax_v.plot(th, vph[0], "k--", lw=1.2)
        sp = spectrum(iph[0]); ok = sp > 1e-6
        ax_s.plot(np.arange(H_MAX)[ok] + offs[-1], sp[ok], "kx", ms=7, label="static optimizer")
        labels, sav, cols = [], [], []
        for k, (name, x, col) in enumerate(zip(HARMONIC_SETS, runs, colors)):
            if x is None or not x.get("ok"):
                continue
            th, iph, vph, _ = dense_waveforms(fwd_plot, w, x["X"])
            ax_i.plot(th, iph[0], color=col, lw=1.3, label=name)
            ax_v.plot(th, vph[0], color=col, lw=1.3)
            sp = spectrum(iph[0]); ok = sp > 1e-6
            ax_s.plot(np.arange(H_MAX)[ok] + offs[k], sp[ok], "o", color=col, ms=4, label=name)
            labels.append(name); sav.append(-100 * (x["copper"] / st["copper"] - 1)); cols.append(col)
        style_wave_axes(ax_i, ax_v); ax_i.legend(fontsize=8)
        style_spectrum(ax_s, rpm); ax_s.legend(fontsize=7, ncol=2)
        bars = ax_c.bar(labels, sav, color=cols)
        ax_c.bar_label(bars, labels=[f"{v:.2f} %" for v in sav], fontsize=8)
        ax_c.set_ylabel("Joule-loss saving vs static [%]"); ax_c.grid(axis="y", alpha=0.3); ax_c.tick_params(axis="x", rotation=15)
        ax_c.set_title("Joule-loss saving vs the static optimizer", fontsize=10)
        finish(fig, os.path.join(wave_dir, f"harmonics_{rpm}rpm_{f * 100:.0f}pct.png"), NOTE_BASE, NOTE_HARM, NOTE_WAVE)

In [ ]:
# Study 2 waveforms: with vs without the 3rd harmonic (static and dynamic), at 80 % of the no-3rd static max
for rpm in [r for r in (1200, 1500, 1800) if r in SPEEDS]:
    w = rpm_to_omega(rpm)
    f = 0.8 if 0.8 in THIRD_FRACS else THIRD_FRACS[0]
    cases = []
    for no3, mode in ((False, "static"), (False, "dynamic"), (True, "static"), (True, "dynamic")):
        x = next((r for r in S2[(mode, no3)] if r["rpm"] == rpm and r["frac"] == f), None)
        cases.append((f"{mode}, {'without' if no3 else 'with'} 3rd harmonic", x))
    if not any(x and x.get("ok") for _, x in cases):
        continue
    fig, (ax_i, ax_v, ax_s) = plt.subplots(1, 3, figsize=(19, 5.2))
    T = next(x["T"] for _, x in cases if x)
    fig.suptitle(f"3rd harmonic, {rpm} rpm, T* = {T:.2f} Nm ({f:.0%} of the static max without 3rd harmonic)", fontsize=13)
    styles = (("tab:blue", "--"), ("tab:blue", "-"), ("tab:red", "--"), ("tab:red", "-"))
    offs = np.linspace(-0.3, 0.3, 4)
    for k, ((lab, x), (col, ls)) in enumerate(zip(cases, styles)):
        if x is None or not x.get("ok"):
            continue
        th, iph, vph, _ = dense_waveforms(fwd_plot, w, x["X"])
        lab2 = f"{lab}: Joule {x['copper']:.0f} A^2"
        ax_i.plot(th, iph[0], ls, color=col, lw=1.4, label=lab2); ax_v.plot(th, vph[0], ls, color=col, lw=1.4)
        sp = spectrum(iph[0]); ok = sp > 1e-6
        ax_s.plot(np.arange(H_MAX)[ok] + offs[k], sp[ok], "o" if ls == "-" else "x", color=col, ms=5, label=lab)
    style_wave_axes(ax_i, ax_v); ax_i.legend(fontsize=7)
    style_spectrum(ax_s, rpm); ax_s.legend(fontsize=7)
    finish(fig, os.path.join(wave_dir, f"third_harmonic_{rpm}rpm.png"), NOTE_BASE, NOTE_THIRD, NOTE_WAVE)

In [ ]:
# Study 4 waveforms: allowed torque ripple band (incl. the torque over the period)
for rpm in FOCUS_SPEEDS:
    w = rpm_to_omega(rpm)
    for f in FOCUS_FRACS:
        st = S4["static"][(rpm, f)]
        runs = sorted([x for x in S4["min"] if x["rpm"] == rpm and x["frac"] == f and x.get("ok")], key=lambda x: x["band"])
        if not runs:
            continue
        fig, ((ax_i, ax_v), (ax_s, ax_t)) = plt.subplots(2, 2, figsize=(16, 9.5))
        T = f * S4["static_max"][rpm]
        fig.suptitle(f"Torque ripple band, {rpm} rpm, T* = {T:.2f} Nm ({f:.0%} of static max), static black dashed", fontsize=13)
        colors = plt.cm.cividis(np.linspace(0.0, 0.9, len(runs)))
        offs = np.linspace(-0.3, 0.3, len(runs) + 1)
        th, iph, vph, Ts = dense_waveforms(fwd_plot, w, st["X"])
        ax_i.plot(th, iph[0], "k--", lw=1.2, label="static"); ax_v.plot(th, vph[0], "k--", lw=1.2); ax_t.plot(th, Ts, "k--", lw=1.2, label="static")
        for k, (x, col) in enumerate(zip(runs, colors)):
            th, iph, vph, Tt = dense_waveforms(fwd_plot, w, x["X"])
            lab = f"band +-{100 * x['band']:g} %: saving {-100 * (x['copper'] / st['copper'] - 1):.2f} %"
            ax_i.plot(th, iph[0], color=col, lw=1.3, label=lab); ax_v.plot(th, vph[0], color=col, lw=1.3)
            ax_t.plot(th, Tt, color=col, lw=1.3, label=f"band +-{100 * x['band']:g} %")
            sp = spectrum(iph[0]); ok = sp > 1e-6
            ax_s.plot(np.arange(H_MAX)[ok] + offs[k], sp[ok], "o", color=col, ms=4, label=f"band +-{100 * x['band']:g} %")
        style_wave_axes(ax_i, ax_v); ax_i.legend(fontsize=7)
        style_spectrum(ax_s, rpm); ax_s.legend(fontsize=7)
        ax_t.axhline(T, color="gray", ls=":"); ax_t.set_xlim(0, 360); ax_t.grid(alpha=0.3)
        ax_t.set_xlabel("rotor angle theta [deg, electrical]"); ax_t.set_ylabel("torque [Nm]"); ax_t.legend(fontsize=7)
        ax_t.set_title("torque over the period (target dotted)", fontsize=10)
        finish(fig, os.path.join(wave_dir, f"torque_band_{rpm}rpm_{f * 100:.0f}pct.png"), NOTE_BASE, NOTE_BAND, NOTE_WAVE)
print("waveform figures saved to", os.path.join(figs_dir, wave_dir))